In [ ]:
# Biological evidence: one shared activity across ten cell identities.
# One cell; upload margin_dictionary_learning.py alongside this notebook in Colab.
# Dependencies: numpy, pandas, scipy, scikit-learn, matplotlib, torch, tqdm.
# CPU acceleration also uses numba and joblib; all fitting limits stay unchanged.
# Kotliar et al. (2019), https://doi.org/10.7554/eLife.43803; adapted scsim model.
# Simplifications: balanced identities, disjoint activity/identity DE genes,
# only upregulation, no doublets or cell QC. This is a generic activity model,
# not a mechanistic senescence model. Parameters follow the preceding pilot.
#
# For identity z_i and activity usage u_i, mu_ig=L_i*((1-u_i)*p_zg+u_i*p_ag).
# Profiles p sum to one; u=0 for inactive cells and Uniform(.1,.7) otherwise.
# Gene variability v_g=22.087/chi-square(22.087); b_ig^2=(.448+mu_ig^-1/2)^2*v_g.
# Rates ~ Gamma(shape=1/b^2, scale=mu*b^2), then counts ~ Poisson(rate).
# Hence E[count]=mu and Var[count | mu,v]=mu+mu^2*b^2.
# A/B hold 600 active cells and their expected activity contribution fixed.
# B adds 3,000 inactive cells in the same ten identities. Carrier identities
# are nested random subsets. A2 has all cells active within its two carriers.
# Preprocessing is learned afresh within each training condition: select 2,000
# variable genes, divide raw counts by training gene SD, then scale the median
# training row norm to one. Apply this same map to truth and held-out cells.
# Recovery jointly matches all eleven atoms by signed cosine, so one atom
# cannot count as both an identity and the activity. Pearson on that same atom
# and cosine to incorrect true identity programs are supporting diagnostics.
# The common held-out panel has 30% active cells within every identity. AP uses
# the matched atom's code divided by observed library size, averaged by identity;
# it includes transfer to identities without active training cells.
#
# Calibration: maximize equally weighted mean matched activity cosine over
# ten conditions and seeds 101-110, separately for each method. Freeze settings
# before seeds 1-50. Figures show the 6,000-cell evaluation settings; all ten
# settings remain in the numerical summaries. Recovery intervals are
# pointwise, unadjusted 95% t intervals across seeds. PR shading shows the
# 10th--90th seed percentiles. Both are conditional on calibrated parameters.

from pathlib import Path
import hashlib
import json
import math
import sys
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from scipy.optimize import linear_sum_assignment
from scipy.stats import t
from sklearn.metrics import average_precision_score, precision_recall_curve
from tqdm.auto import tqdm
from IPython.display import display
from joblib import Parallel, delayed, parallel_config

sys.dont_write_bytecode = True
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'margin_dictionary_learning.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Place margin_dictionary_learning.py in the working folder or an ancestor.')
sys.path.insert(0, str(ROOT))
from margin_dictionary_learning import MarginDictionaryLearning

# All experiment choices are here. Running this cell launches the full study.
RUN_FITS = False  # False: redraw saved results. True: rerun and replace the CSV.
RESUME_FITS = False  # Only with RUN_FITS=True: continue a compatible checkpoint.
N_IDENTITIES, N_A, N_ACTIVE = 10, 3000, 600
CARRIERS = (2, 4, 6, 8, 10)
N_GENES, ACTIVITY_GENES, N_FEATURES = 25000, 1000, 2000
SIGNAL, N_TEST, TEST_PREVALENCE = 1.0, 2000, 0.30
CALIBRATION_SEEDS = tuple(range(101, 111))
EVALUATION_SEEDS = tuple(range(1, 51))
LAMBDA_GRID, Q_GRID = [0.03, 0.04, 0.05], [0.70, 0.75, 0.80]
N_UPDATES, LEARNING_RATE = 5000, 0.01
CODE_TOL, CODE_MAX_ITER, RECORD_EVERY = 1e-5, 5000, 100
DEVICE, BATCH_CELLS = 'auto', 128
METHODS = ('ordinary', 'clipped')
PR_CARRIERS = 6
PR_RECALL = np.linspace(0, 1, 201)
OUTPUT_DIR = ROOT / 'biological_results'
CSV_PATH = OUTPUT_DIR / 'biological_scores.csv'
FIGURE_PATH = OUTPUT_DIR / 'fig_dl_biological_evidence.pdf'
device = torch.device('cuda' if DEVICE == 'auto' and torch.cuda.is_available()
                      else 'cpu' if DEVICE == 'auto' else DEVICE)
CPU_WORKERS = 4
SOLVER = 'gram' if device.type == 'cpu' else 'direct'
torch.set_num_threads(2)  # Data generation; fitting workers each use one thread.
torch.set_float32_matmul_precision('highest')
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

assert N_A % N_IDENTITIES == 0 and N_TEST % N_IDENTITIES == 0
assert all(N_ACTIVE % c == 0 and N_ACTIVE // c <= N_A // N_IDENTITIES for c in CARRIERS)
assert set(CALIBRATION_SEEDS).isdisjoint(EVALUATION_SEEDS)
K = N_IDENTITIES + 1
CONDITIONS = tuple(f'{arm}{c}' for c in CARRIERS for arm in ('A', 'B'))
MODEL_SHA256 = hashlib.sha256((ROOT / 'margin_dictionary_learning.py').read_bytes()).hexdigest()
# The CSV is also the restart checkpoint; no matrices, logs or extra files persist.
protocol = dict(version='biological_v5_cpu_gram', solver=SOLVER, recovery_metric='joint_signed_activity_cosine', pr_carriers=PR_CARRIERS,
                pr_recall_points=len(PR_RECALL), identities=N_IDENTITIES, n_a=N_A, active=N_ACTIVE,
                carriers=CARRIERS, genes=N_GENES, activity_genes=ACTIVITY_GENES,
                features=N_FEATURES, signal=SIGNAL, test_cells=N_TEST,
                test_prevalence=TEST_PREVALENCE, calibration_seeds=CALIBRATION_SEEDS,
                evaluation_seeds=EVALUATION_SEEDS, lambda_grid=LAMBDA_GRID, q_grid=Q_GRID,
                updates=N_UPDATES, lr=LEARNING_RATE,
                code_tol=CODE_TOL, code_max_iter=CODE_MAX_ITER, record_every=RECORD_EVERY,
                batch_cells=BATCH_CELLS, model_sha256=MODEL_SHA256,
                device=str(device), numpy=np.__version__, torch=torch.__version__)
PROTOCOL_JSON = json.dumps(protocol, sort_keys=True)
SETTINGS_ID = hashlib.sha256(PROTOCOL_JSON.encode()).hexdigest()[:16]


def nrng(seed, stream):
    return np.random.default_rng(np.random.SeedSequence([int(seed),int(stream)]))

def trng(seed, stream, block):
    s = int(np.random.SeedSequence([int(seed),int(stream),int(block)])
            .generate_state(1,dtype=np.uint64)[0])
    return torch.Generator(device=device).manual_seed(s)

def normalize(a):
    a = np.asarray(a,dtype=np.float64)
    return a/np.maximum(np.linalg.norm(a,axis=1,keepdims=True),1e-30)

def correlations(a,b):
    return normalize(a-a.mean(1,keepdims=True)) @ normalize(b-b.mean(1,keepdims=True)).T

def sources(seed):
    r=nrng(seed,1)
    base=r.gamma(.34,1/7.68,N_GENES)
    unusual=r.random(N_GENES)<.00286
    base[unusual]=np.median(base)*r.lognormal(6.15,.49,unusual.sum())
    marker=np.zeros(N_GENES,dtype=bool)
    marker[-ACTIVITY_GENES:]=True
    profiles=[]
    for j in range(N_IDENTITIES):
        rr=nrng(seed,10+j)
        de=(rr.random(N_GENES)<.025)&~marker
        p=base.copy()
        p[de]*=np.exp(np.abs(rr.normal(SIGNAL,1.,de.sum())))
        profiles.append(p/p.sum())
    rr=nrng(seed,30)
    p=base.copy()
    p[marker]*=np.exp(np.abs(rr.normal(SIGNAL,1.,marker.sum())))
    profiles.append(p/p.sum())
    noise2=22.087/nrng(seed,31).chisquare(22.087,N_GENES)
    return (torch.tensor(np.array(profiles),device=device,dtype=torch.float64),
            torch.tensor(noise2,device=device,dtype=torch.float64),marker)

def population(seed,stream,n,carriers,nactive):
    r=nrng(seed,stream)
    labels=np.repeat(np.arange(N_IDENTITIES),n//N_IDENTITIES)
    r.shuffle(labels)
    usage=np.zeros(n)
    assert n%N_IDENTITIES==0 and nactive%len(carriers)==0
    for identity in carriers:
        ix=r.choice(np.flatnonzero(labels==identity),nactive//len(carriers),replace=False)
        usage[ix]=r.uniform(.1,.7,len(ix))
    return dict(z=labels,u=usage,L=r.lognormal(7.64,.78,n))

@torch.no_grad()
def sample_counts(seed,stream,pop,profiles,noise2):
    n=len(pop["z"])
    counts=torch.empty((n,N_GENES),device=device,dtype=torch.int32)
    valid=torch.ones((),device=device,dtype=torch.bool)
    for lo in range(0,n,BATCH_CELLS):
        hi=min(lo+BATCH_CELLS,n)
        z=torch.tensor(pop["z"][lo:hi],device=device,dtype=torch.long)
        u=torch.tensor(pop["u"][lo:hi,None],device=device)
        lib=torch.tensor(pop["L"][lo:hi,None],device=device)
        mu=lib*((1-u)*profiles[z]+u*profiles[-1])
        b2=(.448+mu.rsqrt()).square()*noise2
        generator=trng(seed,stream,lo)
        rate=torch._standard_gamma(b2.reciprocal(),generator=generator)*(mu*b2)
        valid &= torch.isfinite(rate).all() & (rate>=0).all()
        safe=torch.nan_to_num(rate,nan=0.,posinf=0.,neginf=0.).clamp_min(0)
        draw=torch.poisson(safe,generator=generator)
        valid &= torch.isfinite(draw).all() & (draw>=0).all() & (draw.max()<2**31-1)
        counts[lo:hi]=draw.to(torch.int32)
    assert bool(valid.item()), "Invalid simulated counts; stop rather than accept them."
    return counts

def concentrate(seed,pop,carriers):
    # Reassign identity backgrounds, preserving every cell's L and u.
    r=nrng(seed,60)
    old=pop["z"]; z=old.copy(); positive=pop["u"]>0
    target=np.zeros(N_IDENTITIES,dtype=int); target[carriers]=N_ACTIVE//len(carriers)
    donors=[]; recipients=[]
    for j in range(N_IDENTITIES):
        excess=int(((old==j)&positive).sum()-target[j])
        if excess>0:
            donors.extend(r.choice(np.flatnonzero((old==j)&positive),excess,replace=False))
        elif excess<0:
            recipients.extend(r.choice(np.flatnonzero((old==j)&~positive),-excess,replace=False))
    r.shuffle(donors); r.shuffle(recipients)
    z[donors]=old[recipients]; z[recipients]=old[donors]
    assert np.array_equal(np.bincount(z,minlength=N_IDENTITIES),np.bincount(old,minlength=N_IDENTITIES))
    assert np.array_equal(np.bincount(z[positive],minlength=N_IDENTITIES),target)
    return dict(z=z,u=pop["u"].copy(),L=pop["L"].copy()),np.flatnonzero(z!=old)

@torch.no_grad()
def preprocess(train,test,profiles):
    # Original-style training-only cNMF feature selection and SD scaling.
    raw=train.double()
    sd=raw.std(0,correction=1)
    keep=((train>0).sum(0)>=math.ceil(len(train)/500))&(sd>0)
    candidates=torch.where(keep)[0]
    assert len(candidates)>=N_FEATURES, "Too few detected variable genes."
    filtered=raw[:,candidates]
    tpm=1e6*filtered/filtered.sum(1,keepdim=True).clamp_min(1)
    variance,mean=torch.var_mean(tpm,dim=0,correction=0)
    mean=mean.cpu().numpy(); variance=variance.cpu().numpy()
    fano=variance/mean
    top=np.argsort(-mean,kind="stable")[:20]
    a=np.min(np.sqrt(variance[top])/mean[top])
    ml,mh=np.quantile(mean,[.1,.9]); fl,fh=np.quantile(fano,[.1,.9])
    box=(mean>ml)&(mean<mh)&(fano>fl)&(fano<fh)
    assert box.any()
    ratio=fano/(a*a*mean+np.median(fano[box]))
    chosen=np.argsort(-ratio,kind="stable")[:N_FEATURES]
    ids=np.sort(candidates.cpu().numpy()[chosen])
    ix=torch.tensor(ids,device=device,dtype=torch.long)
    scale=sd[ix]
    x=train[:,ix].float()/scale.float()
    median=torch.quantile(x.norm(dim=1),.5)
    assert median>0 and torch.isfinite(median)
    scale=scale*median.double()
    x=train[:,ix].float()/scale.float()
    xt=test[:,ix].float()/scale.float()
    truth=normalize((profiles[:,ix]/scale).cpu().numpy())
    assert torch.isfinite(x).all() and torch.isfinite(xt).all()
    return x,xt,truth,ids


def make_weights(q):
    def weights(margins):
        if q is None or q == 1:
            return margins.clamp_min(0)
        if margins.is_cuda:
            positive = margins[margins > 0]
            cap = torch.quantile(positive, q) if positive.numel() else margins.new_zeros(())
        else:
            # The same linearly interpolated quantile, using selection rather than sorting.
            values = margins.numpy()
            positive = values[values > 0]
            if not len(positive):
                return margins.clamp_min(0)
            rank = np.float32(q) * np.float32(len(positive) - 1)
            lower, upper = int(np.floor(rank)), int(np.ceil(rank))
            positive.partition((lower, upper))
            fraction = np.float32(rank - lower)
            difference = positive[upper] - positive[lower]
            cap = (positive[lower] + difference * fraction if fraction < 0.5 else
                   positive[upper] - difference * (np.float32(1) - fraction))
            cap = float(cap)
        return margins.clamp_min(0).clamp_max(cap)
    return weights


def activity_metrics(truth, dictionary):
    # One joint assignment under the thesis's signed cosine metric.
    truth_unit, learned_unit = normalize(truth), normalize(dictionary)
    cosine = truth_unit @ learned_unit.T
    rows, columns = linear_sum_assignment(-cosine)
    assignment = np.empty(K, dtype=int)
    assignment[rows] = columns
    atom = int(assignment[-1])
    pearson = correlations(truth, np.asarray(dictionary, dtype=float))
    return atom, dict(activity_cosine=float(cosine[-1, atom]),
                      best_activity_cosine=float(cosine[-1].max()),
                      identity_cosine=float(cosine[np.arange(N_IDENTITIES),
                                                  assignment[:-1]].mean()),
                      specificity=float(cosine[-1, atom] - cosine[:-1, atom].max()),
                      activity_pearson=float(pearson[-1, atom]),
                      true_identity_activity_cosine=float((truth_unit[:-1] @ truth_unit[-1]).max()))


def datasets(seed):
    # Generate one seed at a time. No all-seed matrix cache is needed.
    profiles, noise2, marker = sources(seed)
    order = nrng(seed, 40).permutation(N_IDENTITIES)
    base = population(seed, 41, N_A, order, N_ACTIVE)
    extra = population(seed, 42, N_A, order, 0)
    raw_base = sample_counts(seed, 100, base, profiles, noise2)
    raw_extra = sample_counts(seed, 101, extra, profiles, noise2)
    test_active = int(round(N_TEST * TEST_PREVALENCE / N_IDENTITIES)) * N_IDENTITIES
    test_pop = population(seed, 43, N_TEST, np.arange(N_IDENTITIES), test_active)
    test = sample_counts(seed, 103, test_pop, profiles, noise2)
    totals = test.double().sum(1).cpu().numpy().clip(1)
    budget = float(np.sum(base['L'] * base['u']))
    for c in CARRIERS:
        carriers = order[:c]
        # Swap active and inactive identity backgrounds, keeping identity sizes,
        # each cell's library budget L, and every positive usage u unchanged.
        pop, changed = concentrate(seed, base, carriers)
        raw_a = raw_base.clone()
        if len(changed):
            subset = {key: value[changed] for key, value in pop.items()}
            raw_a[torch.as_tensor(changed, device=device)] = sample_counts(
                seed, 110 + c, subset, profiles, noise2)
        assert (pop['u'] > 0).sum() == N_ACTIVE
        assert np.isclose(np.sum(pop['L'] * pop['u']), budget, rtol=0, atol=1e-7)
        for arm in ('A', 'B'):
            raw = raw_a if arm == 'A' else torch.cat((raw_a, raw_extra))
            x, xt, truth, ids = preprocess(raw, test, profiles)
            valid = torch.where(x.norm(dim=1) > 0)[0].cpu().numpy()
            initial_ids = nrng(seed, 200 + 2*c + (arm == 'B')).choice(valid, K, replace=False)
            initial = normalize(x[torch.as_tensor(initial_ids, device=device)].cpu().numpy()).astype(np.float32)
            info = dict(n_cells=len(x), n_active=N_ACTIVE, carriers=c, arm=arm,
                        global_prevalence=N_ACTIVE/len(x),
                        within_carrier_prevalence=N_ACTIVE/(c*len(x)/N_IDENTITIES),
                        expected_activity_counts=budget,
                        retained_activity_genes=int(marker[ids].sum()),
                        initial_sha256=hashlib.sha256(initial.tobytes()).hexdigest())
            yield f'{arm}{c}', x, xt, truth, initial, info, test_pop, totals, carriers
            del raw, x, xt, initial
        del raw_a


def macro_precision_curve(test_pop, scores):
    # Step-sample each identity's empirical PR curve on the same recall grid.
    # Average identities within a seed; independent seeds remain separate.
    curves = []
    for identity in range(N_IDENTITIES):
        keep = test_pop['z'] == identity
        labels = test_pop['u'][keep] > 0
        precision, recall, _ = precision_recall_curve(labels, scores[keep])
        indices = np.searchsorted(recall[::-1], PR_RECALL, side='left')
        curve = precision[::-1][indices]
        curve[-1] = labels.mean()  # Endpoint: classify every cell as active.
        curves.append(curve)
    return np.mean(curves, axis=0)


def precision_lower_limit(*displayed):
    # Start at .25 and extend down by .05 until every displayed curve and band fits.
    values = np.concatenate([np.asarray(value, dtype=float).ravel() for value in displayed])
    if not np.isfinite(values).all() or (values < 0).any() or (values > 1).any():
        raise ValueError('Displayed precision values must be finite and between zero and one.')
    lower = .25
    while values.min() < lower and lower > 0:
        lower = round(lower - .05, 2)
    return lower


def plot_precision_recall(evaluation):
    colors = {'ordinary': '#C47A44', 'clipped': '#2678AC'}
    curves_by_method, displayed = {}, []
    for method in METHODS:
        rows = evaluation[(evaluation.condition == f'B{PR_CARRIERS}')
                          & (evaluation.method == method)].sort_values('seed')
        assert tuple(rows.seed.astype(int)) == tuple(sorted(EVALUATION_SEEDS))
        curves = np.array([json.loads(value) for value in rows.pr_precision_json])
        assert curves.shape == (len(EVALUATION_SEEDS), len(PR_RECALL))
        mean = curves.mean(axis=0)
        low, high = np.quantile(curves, [.1, .9], axis=0)
        curves_by_method[method] = mean, low, high
        displayed.extend((mean, low, high))
    # Shading shows between-seed 10th--90th percentiles, not a confidence interval.
    # Include the prevalence reference when choosing the lower limit.
    lower = precision_lower_limit(*displayed, [TEST_PREVALENCE])
    with plt.rc_context():
        plt.rcParams.update({
            'font.family': 'STIXGeneral', 'mathtext.fontset': 'stix', 'font.size': 10,
            'axes.labelsize': 10, 'axes.titlesize': 10, 'xtick.labelsize': 9, 'ytick.labelsize': 9,
            'axes.linewidth': .6, 'axes.edgecolor': '#8C939A',
            'xtick.color': '#39434A', 'ytick.color': '#39434A',
            'text.color': '#263238', 'axes.labelcolor': '#263238',
            'axes.spines.top': False, 'axes.spines.right': False, 'pdf.fonttype': 42})
        fig, ax = plt.subplots(figsize=(4.8, 3.3))
        ax.set_axisbelow(True)
        ax.grid(axis='y', color='#E8EBEE', linewidth=.6)
        ax.tick_params(length=3, width=.6, pad=5)
        ax.spines['left'].set_color('#A4ABB1')
        ax.spines['bottom'].set_color('#A4ABB1')
        for method in METHODS:
            mean, low, high = curves_by_method[method]
            ax.fill_between(PR_RECALL, low, high, color=colors[method], alpha=.14, linewidth=0)
            ax.plot(PR_RECALL, mean, color=colors[method], linewidth=1.8)
        ax.axhline(TEST_PREVALENCE, color='#929BA3', linewidth=.9, linestyle=(0, (3, 3)))
        ax.set(xlabel='Recall', xlim=(0, 1), ylim=(lower, 1.02))
        ax.set_ylabel('Precision')
        fig.subplots_adjust(left=.14, right=.98, bottom=.18, top=.97)
        fig.savefig(OUTPUT_DIR / 'fig_dl_biological_precision_recall.pdf',
                    bbox_inches='tight', pad_inches=.08)
        plt.show()
        plt.close(fig)


def fit_key(stage, seed, condition, method, lam, q):
    return stage, int(seed), condition, method, round(float(lam), 10), None if pd.isna(q) else round(float(q), 10)


def save_records():
    # Atomic replacement protects the previous checkpoint if writing is interrupted.
    temporary = CSV_PATH.with_suffix('.csv.tmp')
    pd.DataFrame(records).to_csv(temporary, index=False)
    # Windows readers can briefly prevent replacing an otherwise valid checkpoint.
    for attempt in range(100):
        try:
            temporary.replace(CSV_PATH)
            return
        except PermissionError:
            if attempt == 99:
                raise
            time.sleep(0.1)


def run_fit(stage, seed, condition, method, lam, q, data):
    x, xt, truth, initial, info, test_pop, totals, carriers = data
    # Each CPU fit uses one numerical thread; preserve the parent setting in serial mode.
    previous_threads = torch.get_num_threads()
    if device.type == 'cpu':
        torch.set_num_threads(1)
    start = time.perf_counter()
    row = dict(stage=stage, seed=seed, condition=condition, method=method,
               alpha=lam, q=q, settings_id=SETTINGS_ID, protocol_json=PROTOCOL_JSON, **info)
    try:
        def callback(iteration, dictionary):
            _, metrics = activity_metrics(truth, dictionary)
            return dict(activity_cosine=metrics['activity_cosine'])
        model = MarginDictionaryLearning(
            K, make_weights(q), alpha=lam, lr=LEARNING_RATE, n_iter=N_UPDATES,
            code_tol=CODE_TOL, code_max_iter=CODE_MAX_ITER, dict_init=initial.copy(),
            device=str(device), solver=SOLVER, record_every=RECORD_EVERY, verbose=False,
            callback=callback).fit(x)
        atom, metrics = activity_metrics(truth, model.components_)
        tail = model.history_[-(500 // RECORD_EVERY + 1):]
        row.update(status='ok', **metrics, n_updates=model.n_iter_,
                   final_atom_change=model.history_[-1]['max_atom_change'],
                   activity_cosine_range_last_500=float(np.ptp([h['activity_cosine'] for h in tail])),
                   max_atom_change_last_500=max(h['max_atom_change'] for h in tail),
                   **model.final_coding_diagnostics_)
        # Test scores never influence calibration. Truth is used only to identify
        # the matched atom for simulation evaluation, not to fit the dictionary.
        if stage == 'evaluation':
            scores = model.transform(xt)[:, atom] / totals
            if len(carriers) == PR_CARRIERS:
                row['pr_precision_json'] = json.dumps(macro_precision_curve(test_pop, scores).tolist())
            ap = np.array([average_precision_score(test_pop['u'][test_pop['z'] == j] > 0,
                                                   scores[test_pop['z'] == j])
                           for j in range(N_IDENTITIES)])
            row.update(test_ap=float(ap.mean()), supported_test_ap=float(ap[carriers].mean()),
                       test_prevalence=float((test_pop['u'] > 0).mean()),
                       **{f'test_{key}': value for key, value in model.transform_diagnostics_.items()})
    except Exception as exc:
        row.update(status='failed', error=f'{type(exc).__name__}: {exc}')
    finally:
        if device.type == 'cpu' and previous_threads != 1:
            torch.set_num_threads(previous_threads)
    return dict(row, seconds=time.perf_counter() - start)


def run_stage(stage, seeds, candidates):
    missing = {(seed, condition, method, lam, q)
               for seed in seeds for condition in CONDITIONS for method, lam, q in candidates
               if fit_key(stage, seed, condition, method, lam, q) not in completed}

    def jobs():
        for seed in seeds:
            if not any(job[0] == seed for job in missing):
                continue
            for condition, *data in datasets(seed):
                # NumPy arrays allow temporary shared memory maps between CPU workers.
                data = tuple(v.numpy() if isinstance(v, torch.Tensor) and not v.is_cuda else v
                             for v in data)
                for method, lam, q in candidates:
                    if (seed, condition, method, lam, q) in missing:
                        yield delayed(run_fit)(stage, seed, condition, method, lam, q, data)

    workers = CPU_WORKERS if device.type == 'cpu' else 1
    with parallel_config(backend='loky', inner_max_num_threads=1):
        with Parallel(n_jobs=workers, return_as='generator_unordered',
                      pre_dispatch='2*n_jobs', batch_size=1, mmap_mode='c') as pool:
            results = pool(jobs())
            try:
                with tqdm(total=len(missing), desc=stage.capitalize(), unit='fit') as progress:
                    for row in results:
                        # Only the parent writes the checkpoint; completion order is irrelevant.
                        records.append(row)
                        save_records()
                        if row['status'] != 'ok':
                            raise RuntimeError(f"{row['condition']}, seed {row['seed']}: {row['error']}")
                        completed.add(fit_key(stage, row['seed'], row['condition'], row['method'],
                                              row['alpha'], None if row['method'] == 'ordinary' else row['q']))
                        progress.set_postfix_str(f"{row['condition']}, {row['method']}, seed {row['seed']}",
                                                 refresh=False)
                        progress.update(1)
            finally:
                results.close()


def select_parameters(candidates):
    # Replay only the current search round on resume. Later rounds may be partial.
    allowed = {(method, round(lam, 10), q) for method, lam, q in candidates}
    calibration = pd.DataFrame([row for row in records
                               if row['stage'] == 'calibration'
                               and (row['method'], round(row['alpha'], 10),
                                    None if pd.isna(row['q']) else round(row['q'], 10)) in allowed])
    expected = {(seed, condition) for seed in CALIBRATION_SEEDS for condition in CONDITIONS}
    for _, group in calibration.groupby(['method', 'alpha', 'q'], dropna=False):
        assert len(group) == len(expected)
        assert set(zip(group.seed, group.condition)) == expected
        assert group.status.eq('ok').all() and np.isfinite(group.activity_cosine).all()
    assert calibration.groupby(['seed', 'condition']).initial_sha256.nunique().eq(1).all()
    calibration = calibration.sort_values(['seed', 'condition', 'method', 'alpha', 'q'])
    scores = calibration.groupby(['method', 'alpha', 'q'], dropna=False).agg(
        mean_cosine=('activity_cosine', 'mean'), n=('activity_cosine', 'count')).reset_index()
    assert scores.n.eq(len(CALIBRATION_SEEDS) * len(CONDITIONS)).all()
    # Complete, balanced groups make this the equal-condition, equal-seed mean.
    ranked = scores.sort_values(['mean_cosine', 'alpha', 'q'], ascending=[False, True, False])
    selected = {method: (float(group.iloc[0].alpha),
                        None if method == 'ordinary' else float(group.iloc[0].q))
                for method, group in ranked.groupby('method')}
    return selected, scores.groupby('method').mean_cosine.max().to_dict(), scores


def calibrate():
    lambdas, quantiles = sorted(LAMBDA_GRID), sorted(Q_GRID)
    previous_best = None
    while True:
        candidates = [('ordinary', lam, None) for lam in lambdas]
        candidates += [('clipped', lam, q) for lam in lambdas for q in quantiles]
        run_stage('calibration', CALIBRATION_SEEDS, candidates)
        selected, best, scores = select_parameters(candidates)
        improved = {method: previous_best is None or best[method] > previous_best[method]
                    for method in METHODS}
        extra_lambdas, extra_q = set(), set()
        for method, (lam, _) in selected.items():
            if improved[method]:
                if lam == min(lambdas) and lam > 0.01:
                    extra_lambdas.add(round(lam - 0.01, 10))
                if lam == max(lambdas):
                    extra_lambdas.add(round(lam + 0.01, 10))
        q = selected['clipped'][1]
        # Same boundary-extension rule as the bars calibration notebook.
        if improved['clipped']:
            if q == min(quantiles) and q > 0:
                extra_q.add(round(max(0, q - 0.05), 10))
            if q == max(quantiles) and q < 1:
                extra_q.add(round(min(1, q + 0.05), 10))
        extra_lambdas -= set(lambdas)
        extra_q -= set(quantiles)
        if not extra_lambdas and not extra_q:
            return selected, scores
        lambdas = sorted(set(lambdas) | extra_lambdas)
        quantiles = sorted(set(quantiles) | extra_q)
        previous_best = best


def mean_interval(values):
    values = np.asarray(values, dtype=float)
    assert len(values) == len(EVALUATION_SEEDS) and np.isfinite(values).all()
    mean = float(values.mean())
    half = float(t.ppf(.975, len(values)-1) * values.std(ddof=1)/np.sqrt(len(values)))
    return mean, half


def summarize_and_plot():
    frame = pd.DataFrame(records)
    evaluation = frame.query("stage == 'evaluation'").copy()
    expected = {(seed, condition, method) for seed in EVALUATION_SEEDS
                for condition in CONDITIONS for method in METHODS}
    assert set(zip(evaluation.seed, evaluation.condition, evaluation.method)) == expected
    assert len(evaluation) == len(expected) and evaluation.status.eq('ok').all()
    assert evaluation.groupby(['seed', 'condition']).initial_sha256.nunique().eq(1).all()
    for method in METHODS:
        rows = evaluation[evaluation.method == method]
        assert rows.alpha.nunique() == 1 and rows.q.nunique(dropna=False) == 1
    table = evaluation.groupby(['condition', 'method'], sort=False).agg(
        cells=('n_cells', 'first'), carriers=('carriers', 'first'),
        carrier_prevalence=('within_carrier_prevalence', 'first'),
        alpha=('alpha', 'first'), q=('q', 'first'),
        activity_cosine=('activity_cosine', 'mean'), activity_sd=('activity_cosine', 'std'),
        best_atom_cosine=('best_activity_cosine', 'mean'), specificity=('specificity', 'mean'),
        activity_pearson=('activity_pearson', 'mean'),
        identity_reference_cosine=('true_identity_activity_cosine', 'mean'),
        test_ap=('test_ap', 'mean'), identity_cosine=('identity_cosine', 'mean'),
        maximum_final_movement=('final_atom_change', 'max'),
        maximum_tail_cosine_range=('activity_cosine_range_last_500', 'max'))
    display(table.round(4))
    # Paired dilution and interaction estimates use entire seed vectors.
    wide = evaluation.pivot(index='seed', columns=['method', 'condition'], values='activity_cosine')
    contrasts = []
    for method in METHODS:
        reference = wide[method, 'B10'] - wide[method, 'A10']
        for c in CARRIERS:
            delta = wide[method, f'B{c}'] - wide[method, f'A{c}']
            mean, half = mean_interval(delta)
            interaction, interaction_half = mean_interval(delta - reference)
            contrasts.append(dict(method=method, carriers=c, dilution=mean, half95=half,
                                  interaction_vs_10=interaction, interaction_half95=interaction_half))
    contrasts = pd.DataFrame(contrasts)
    # Recovery: 6,000 training cells, with seed dispersion and mean trends.
    with plt.rc_context():
        plt.rcParams.update({
            'font.family': 'STIXGeneral', 'mathtext.fontset': 'stix', 'font.size': 10,
            'axes.labelsize': 10, 'axes.titlesize': 10, 'xtick.labelsize': 9, 'ytick.labelsize': 9,
            'axes.linewidth': .6, 'axes.edgecolor': '#8C939A',
            'xtick.color': '#39434A', 'ytick.color': '#39434A',
            'text.color': '#263238', 'axes.labelcolor': '#263238',
            'axes.spines.top': False, 'axes.spines.right': False, 'pdf.fonttype': 42})
        fig, ax = plt.subplots(figsize=(4.8, 3.3))
        colors = {'ordinary': '#C47A44', 'clipped': '#2678AC'}
        jitter = np.random.default_rng(0)  # Plot-only jitter; never affects fitting.
        minimum, maximum = -.025, 1.025
        ax.set_axisbelow(True)
        ax.grid(axis='y', color='#E8EBEE', linewidth=.6)
        ax.tick_params(length=3, width=.6, pad=5)
        ax.spines['left'].set_color('#A4ABB1')
        ax.spines['bottom'].set_color('#A4ABB1')
        for method, offset in [('ordinary', -.20), ('clipped', .20)]:
            means, halves = [], []
            for c in CARRIERS:
                values = evaluation.loc[
                    (evaluation.method == method) & (evaluation.condition == f'B{c}'),
                    'activity_cosine'].to_numpy()
                mean, half = mean_interval(values)
                means.append(mean)
                halves.append(half)
                minimum = min(minimum, values.min()-.025, mean-half-.025)
                maximum = max(maximum, values.max()+.025, mean+half+.025)
                ax.scatter(c+offset+jitter.uniform(-.12, .12, len(values)), values,
                           s=12, color=colors[method], alpha=.30,
                           edgecolors='white', linewidths=.25, zorder=2)
            ax.errorbar(np.asarray(CARRIERS)+offset, means, yerr=halves,
                        fmt='o-', color=colors[method], markersize=5.5,
                        markeredgecolor='white', markeredgewidth=.8,
                        linewidth=1.2, elinewidth=1.1, capsize=2.5, zorder=4)
        ax.set(xlabel='Identities carrying the activity',
               xticks=CARRIERS, xlim=(min(CARRIERS)-.85, max(CARRIERS)+.85))
        ax.set(ylabel='Matched activity cosine', ylim=(minimum, maximum))
        fig.subplots_adjust(left=.14, right=.98, bottom=.18, top=.97)
        fig.savefig(FIGURE_PATH, bbox_inches='tight', pad_inches=.08)
        plt.show()
        plt.close(fig)
    plot_precision_recall(evaluation)
    return table, contrasts


# Resume only the exact protocol. A failed fit remains visible and blocks
# inference; resolve its cause before rerunning rather than silently dropping it.
records = (pd.read_csv(CSV_PATH).to_dict('records')
           if CSV_PATH.exists() and (not RUN_FITS or RESUME_FITS) else [])
if records:
    if RUN_FITS and {row['settings_id'] for row in records} != {SETTINGS_ID}:
        raise ValueError('Cannot resume this checkpoint with the current code/settings. '
                         'Set RESUME_FITS=False for a fresh run, or RUN_FITS=False to redraw saved results.')
    if not RUN_FITS:
        saved_protocols = {row['protocol_json'] for row in records}
        assert len(saved_protocols) == 1, 'The CSV mixes experiment protocols.'
        saved = json.loads(next(iter(saved_protocols)))
        requested = json.loads(PROTOCOL_JSON)
        for key in ('device', 'numpy', 'torch', 'model_sha256', 'solver'):
            saved.pop(key, None)
            requested.pop(key, None)
        if saved != requested:
            raise ValueError('Plot settings differ from the saved experiment; restore the CSV settings above.')
    if any(row['status'] != 'ok' for row in records):
        raise RuntimeError('The CSV contains a failed fit. Inspect its error before continuing.')
completed = {fit_key(row['stage'], row['seed'], row['condition'], row['method'], row['alpha'], row['q'])
             for row in records}
assert len(completed) == len(records), 'Duplicate fits in the score CSV.'
started = time.perf_counter()
if RUN_FITS:
    OUTPUT_DIR.mkdir(exist_ok=True)
    with torch.no_grad():
        selected, calibration_scores = calibrate()
        calibration_scores['selected'] = [
            (float(row.alpha), None if row.method == 'ordinary' else float(row.q)) == selected[row.method]
            for row in calibration_scores.itertuples()]
        print('Calibration: equal-condition mean activity cosine; marked settings frozen for evaluation.')
        display(calibration_scores.sort_values(['method', 'alpha', 'q']).round(5))
        for row in records:
            lam, q = selected[row['method']]
            row.update(selected_alpha=lam, selected_q=q, selection_frozen=True)
        save_records()
        run_stage('evaluation', EVALUATION_SEEDS,
                  [(method, *selected[method]) for method in METHODS])
    for row in records:
        lam, q = selected[row['method']]
        row.update(selected_alpha=lam, selected_q=q, selection_frozen=True)
    save_records()
elif not records:
    raise FileNotFoundError('No scores yet; set RUN_FITS=True to generate the CSV.')
print(f'{device} | {N_UPDATES:,} updates; {len(EVALUATION_SEEDS)} evaluation seeds; '
      f'{(time.perf_counter()-started)/60:.1f} min this session. '
      'Recovery: pointwise 95% intervals; PR shading: between-seed 10th-90th percentiles.')
summary, paired_contrasts = summarize_and_plot()
